# Gemini 내장 도구 사용하기

Gemini의 내장 도구(Built-in Tools)를 사용하면 검색이나 코드 실행 기능을 직접 구현하지 않아도 됩니다. Gemini가 필요한 도구를 선택하고 Google 서버에서 실행한 뒤, 실행 결과를 활용해 최종 답변을 만듭니다.

이 노트북에서는 다음 내장 도구를 사용합니다.

- `code_execution`: Python 코드를 생성하고 실행합니다.
- `google_search`: Google 검색 결과를 바탕으로 답변합니다.

공식 문서: https://ai.google.dev/gemini-api/docs/tools

## 실습 환경 준비

In [1]:
import json
import os

from dotenv import load_dotenv
from google import genai

load_dotenv()
api_key = os.getenv("GEMINI_API_KEY")
if not api_key:
    raise ValueError(".env 파일에 GEMINI_API_KEY를 설정해 주세요.")

model = os.getenv("GEMINI_MODEL", "gemini-3.6-flash")
client = genai.Client(api_key=api_key)
print("준비 완료 / 사용 모델:", model)

준비 완료 / 사용 모델: gemini-3.6-flash


## 내장 도구와 사용자 정의 도구 비교

| 구분 | 내장 도구 | 사용자 정의 Function Calling |
| --- | --- | --- |
| 제공자 | Google | 개발자 |
| 도구 정의 | 도구 종류만 등록 | 함수 설명과 매개변수 schema 작성 |
| 실행 위치 | Gemini 서버 | 사용자 애플리케이션 |
| 결과 전달 | Gemini가 자동 처리 | 애플리케이션이 `function_result` 전달 |
| 활용 예 | 웹 검색, 코드 실행, 지도, 파일 검색 | 주문 조회, 사내 API, 예약 처리 |

내장 도구도 `tools`에 등록하지만, Python 함수를 직접 실행하거나 실행 결과를 다시 전달할 필요가 없습니다.

## Code Execution으로 계산하기

`code_execution`을 등록하면 Gemini가 문제 해결에 필요한 Python 코드를 만들고 실행할 수 있습니다. 단순히 코드를 작성해 주는 것이 아니라, 서버의 실행 환경에서 실제로 실행합니다.

In [2]:
code_interaction = client.interactions.create(
    model=model,
    input=(
        "1부터 100까지의 자연수 중 3의 배수이면서 5의 배수인 수를 "
        "Python으로 구하고, 그 합도 알려줘."
    ),
    tools=[{"type": "code_execution"}],
)

print(code_interaction.output_text)

1부터 100까지의 자연수 중 3의 배수이면서 5의 배수인 수(즉, **15의 배수**)와 그 합을 구하는 Python 코드 및 결과입니다.

### Python 코드

```python
# 1부터 100까지의 자연수 중 3의 배수이면서 5의 배수인 수 구하기
numbers = [i for i in range(1, 101) if i % 3 == 0 and i % 5 == 0]

# 해당 수들의 합 구하기
total_sum = sum(numbers)

print("3의 배수이면서 5의 배수인 수:", numbers)
print("수의 합:", total_sum)
```

---

### 실행 결과

* **조건을 만족하는 수**: `15, 30, 45, 60, 75, 90` (총 6개)
* **해당 수들의 합**: **`315`**


### 코드 실행 과정 확인하기

Interaction의 `steps`에는 모델의 답변뿐 아니라 코드 실행 요청과 실행 결과도 순서대로 들어 있습니다.

In [3]:
for step in code_interaction.steps:
    print("step type:", step.type)

    if step.type == "code_execution_call":
        print("실행 코드:")
        print(step.arguments.code)
    elif step.type == "code_execution_result":
        print("실행 결과:")
        print(step.result)

    print("-" * 50)

step type: thought
--------------------------------------------------
step type: code_execution_call
실행 코드:
numbers = [i for i in range(1, 101) if i % 3 == 0 and i % 5 == 0]
total_sum = sum(numbers)

print(f"조건을 만족하는 수: {numbers}")
print(f"합계: {total_sum}")

--------------------------------------------------
step type: code_execution_result
실행 결과:
조건을 만족하는 수: [15, 30, 45, 60, 75, 90]
합계: 315

--------------------------------------------------
step type: thought
--------------------------------------------------
step type: model_output
--------------------------------------------------


## Google Search로 최신 정보 찾기

모델이 학습한 지식만으로는 오늘의 날씨, 최근 뉴스처럼 계속 바뀌는 정보를 정확히 답하기 어렵습니다. `google_search`를 등록하면 Gemini가 필요한 검색어를 만들고 검색 결과를 근거로 답변합니다.

In [ ]:
search_interaction = client.interactions.create(
    model=model,
    input=(
        "오늘 기준으로 Python 공식 홈페이지에 공개된 최신 안정 버전을 찾아서 "
        "버전과 출시일을 알려줘. 출처도 함께 제시해 줘."
    ),
    tools=[
        {
            "type": "google_search",
            "search_types": ["web_search"],
        }
    ],
)

print(search_interaction.output_text)

### 검색 과정 확인하기

검색을 사용한 Interaction에는 `google_search_call`, `google_search_result`, `model_output` 등의 Step이 포함될 수 있습니다.

In [ ]:
for step in search_interaction.steps:
    print("step type:", step.type)

    if step.type == "google_search_call":
        print("검색 요청:", step.arguments)
    elif step.type == "google_search_result":
        print("검색 결과를 받았습니다.")

    print("-" * 50)

### 답변의 출처 확인하기

검색을 바탕으로 생성된 텍스트에는 출처 정보가 `annotations`로 포함될 수 있습니다. 다음 코드는 모델 출력에서 URL 인용 정보를 찾습니다.

In [ ]:
for step in search_interaction.steps:
    if step.type != "model_output":
        continue

    for block in step.content:
        annotations = getattr(block, "annotations", None) or []
        for annotation in annotations:
            if annotation.type == "url_citation":
                print("제목:", annotation.title)
                print("URL:", annotation.url)
                print()

## 여러 내장 도구 함께 사용하기

하나의 요청에 여러 내장 도구를 등록할 수도 있습니다. 다음 예제에서 Gemini는 최신 정보를 검색하고, 계산이 필요하면 Python 코드를 실행할 수 있습니다.

In [ ]:
combined_interaction = client.interactions.create(
    model=model,
    input=(
        "오늘 기준 원/달러 환율을 검색한 뒤, 150달러가 몇 원인지 계산해 줘. "
        "사용한 환율과 출처도 알려줘."
    ),
    tools=[
        {"type": "google_search", "search_types": ["web_search"]},
        {"type": "code_execution"},
    ],
)

print(combined_interaction.output_text)
print("\n도구 실행 순서")
for step in combined_interaction.steps:
    print(step.type)

### 내장 도구와 커스텀 Function Calling 함께 사용하기

Gemini 3 모델에서는 내장 도구와 커스텀 함수를 하나의 `tools` 목록에 등록할 수 있습니다. 내장 도구는 Gemini 서버에서 자동으로 실행되지만, 커스텀 함수는 애플리케이션에서 실행한 뒤 `function_result`를 전달해야 합니다.

다음 예제는 Google Search로 현재 서울 날씨를 확인하고, 커스텀 함수를 여러 번 호출해 매장의 우산과 우비 재고를 각각 조회합니다.

In [ ]:
STORE_STOCK = {
    "우산": 8,
    "우비": 3,
}

def get_store_stock(product_name: str) -> dict:
    """상품명으로 매장 재고를 조회합니다."""
    stock = STORE_STOCK.get(product_name)
    if stock is None:
        return {"ok": False, "error": "상품을 찾을 수 없습니다."}
    return {"ok": True, "product_name": product_name, "stock": stock}

get_store_stock_tool = {
    "type": "function",
    "name": "get_store_stock",
    "description": "상품명으로 오프라인 매장의 현재 재고를 조회합니다.",
    "parameters": {
        "type": "object",
        "properties": {
            "product_name": {
                "type": "string",
                "description": "조회할 상품명. 예: 우산",
            }
        },
        "required": ["product_name"],
    },
}

In [ ]:
mixed_tools = [
    {"type": "google_search", "search_types": ["web_search"]},
    get_store_stock_tool,
]

mixed_interaction = client.interactions.create(
    model=model,
    input=(
        "오늘 서울 날씨를 검색하고 비가 오는지 알려줘. "
        "우리 매장의 우산과 우비 재고를 각각 확인해서 무엇을 준비할 수 있는지 답해 줘."
    ),
    tools=mixed_tools,
    store=True,
)

function_calls = []
for step in mixed_interaction.steps:
    print("step type:", step.type)
    if step.type == "function_call":
        function_calls.append(step)

if not function_calls:
    print(mixed_interaction.output_text)
else:
    function_results = []

    for function_call in function_calls:
        stock_result = get_store_stock(**function_call.arguments)
        function_results.append(
            {
                "type": "function_result",
                "name": function_call.name,
                "call_id": function_call.id,
                "result": [
                    {
                        "type": "text",
                        "text": json.dumps(stock_result, ensure_ascii=False),
                    }
                ],
            }
        )

    final_mixed_interaction = client.interactions.create(
        model=model,
        input=function_results,
        tools=mixed_tools,
        previous_interaction_id=mixed_interaction.id,
        store=True,
    )
    print(final_mixed_interaction.output_text)

### 실행 흐름

`Google Search 자동 실행` → `여러 function_call 반환` → `애플리케이션에서 각 함수 실행` → `function_result 목록 전달` → `최종 답변`

한 응답에 `function_call`이 여러 개 포함될 수 있으므로 리스트에 모두 저장하고, 각 호출의 `call_id`와 일치하는 결과를 전달합니다. `previous_interaction_id`를 전달하면 첫 번째 요청에서 실행된 Google Search의 문맥과 커스텀 함수 결과가 이어집니다.

## 정리

내장 도구의 전체 흐름은 다음과 같습니다.

`사용자 질문` → `Gemini가 도구 선택` → `Gemini 서버에서 도구 실행` → `실행 결과를 활용한 최종 답변`

사용자 정의 Function Calling과 달리, 내장 도구는 개발자가 함수를 실행하거나 `function_result`를 전달하지 않습니다. 다만 모든 질문에 도구가 필요한 것은 아니므로, 필요한 도구만 등록하고 `steps`와 출처를 통해 실제 사용 여부와 근거를 확인해야 합니다.

## 실습 문제 1: Code Execution

`code_execution`을 사용해 다음 문제를 해결하세요.

> 피보나치 수열의 처음 20개 항을 구하고, 그중 짝수인 항의 합을 계산해 줘.

실행 결과뿐 아니라 `steps`에서 Gemini가 만든 Python 코드도 출력하세요.

In [ ]:
# TODO: code_execution 도구를 등록하고 문제를 해결해 보세요.


# TODO: 코드 실행 요청과 결과를 출력해 보세요.

## 실습 문제 2: Google Search

`google_search`를 사용해 현재 서울의 날씨를 검색하고, 외출할 때 적절한 옷차림을 물어보세요.

최종 답변과 함께 모델 출력에 포함된 URL 출처도 출력하세요.

In [ ]:
# TODO: google_search 도구를 등록하고 현재 날씨를 검색해 보세요.


# TODO: URL 인용 정보를 출력해 보세요.

## 실습 문제 3: 두 도구 활용

`google_search`와 `code_execution`을 함께 등록하고 다음 요청을 실행하세요.

> 현재 비트코인 원화 가격을 검색하고, 비트코인 0.015개의 가격을 원 단위로 계산해 줘. 사용한 가격과 출처도 알려줘.

마지막으로 `steps`를 출력해 두 도구가 실제로 사용되었는지 확인하세요.

In [ ]:
# TODO: google_search와 code_execution을 함께 등록하세요.


# TODO: 최종 답변과 전체 Step의 type을 출력하세요.